Install the pandas package.

In [1]:
%pip install pandas


Note: you may need to restart the kernel to use updated packages.


This loads the summary statistics file from the Fulford study into a pandas dataframe and inspects the dimensions and first few rows.

In [6]:
import pandas as pd
file_path = "/Users/aanikaschueler/Desktop/Beiwe/beiwe/code/forest_mano/Data Volumes and Summary Statistics - BU_Fulford_ Smartphone sensing of social activity clinical - 2026-06-26 21_54 (UTC).csv"
df = pd.read_csv(file_path)
print(df.shape)
print(df.head())

(50507, 68)
         Date Participant Id                  Study Id Timezone  \
0  2022-07-06       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
1  2022-07-07       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
2  2022-07-08       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
3  2022-07-09       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   
4  2022-07-10       163lrwb6  UZ3mQzoQGo2TPEzWYw2z5dTT      EDT   

   Accelerometer Bytes  Ambient Audio Bytes  App Log Bytes  Bluetooth Bytes  \
0                  NaN                  NaN            NaN              NaN   
1                  NaN                  NaN            NaN              NaN   
2                  NaN                  NaN            NaN              NaN   
3                  NaN                  NaN            NaN              NaN   
4                  NaN                  NaN            NaN              NaN   

   Calls Bytes  Devicemotion Bytes  ...  Outgoing Call Degree  \
0          NaN                 NaN  ...      

This defines the mapping between each raw Beiwe data stream and the summary statistics that should be generated from that stream. The mappings are used to validate whether summary metrics are present when corresponding raw data exists.

In [7]:
STREAMS = {
    "GPS": {
        "raw_col": "Gps Bytes",
        "metric_cols": [
            "Distance Diameter",
            "Distance From Home",
            "Distance Traveled",
            "Flight Distance Average",
            "Flight Distance Stddev",
            "Flight Duration Average",
            "Flight Duration Stddev",
            "Home Duration",
            "Gyration Radius",
            "Significant Location Count",
            "Significant Location Entropy",
            "Pause Time",
            "Obs Duration",
            "Obs Day",
            "Obs Night",
            "Total Flight Time",
            "Av Pause Duration",
            "Sd Pause Duration",
        ],
    },
    "Accelerometer": {
        "raw_col": "Accelerometer Bytes",
        "metric_cols": ["Walking Time", "Steps", "Cadence"],
    },
    "Calls": {
        "raw_col": "Calls Bytes",
        "metric_cols": [
            "Incoming Call Count",
            "Incoming Call Degree",
            "Incoming Call Duration",
            "Outgoing Call Count",
            "Outgoing Call Degree",
            "Outgoing Call Duration",
            "Missed Call Count",
            "Missed Callers",
        ],
    },
    "Texts": {
        "raw_col": "Texts Bytes",
        "metric_cols": [
            "Incoming Text Count",
            "Incoming Text Degree",
            "Incoming Text Length",
            "Outgoing Text Count",
            "Outgoing Text Degree",
            "Outgoing Text Length",
            "Incoming Text Reciprocity",
            "Outgoing Text Reciprocity",
            "Outgoing Mms Count",
            "Incoming Mms Count",
        ],
    },
}

This creates a validation function that compares raw data availability against summary statistic availability for a given data stream. Specifically, this function identifies days with raw data present (raw_col > 0) and checks if all expected summary metrics are available. Then, it returns a report containing the participant ID, date, stream name, and issue type if there are any flags.

In [8]:
def check_stream(df, stream_name, raw_col, metric_cols):
    raw_present = df[raw_col].fillna(0) > 0

    metrics_present = df[metric_cols].notna().all(axis=1)

    issue = pd.Series([None] * len(df), index=df.index)

    issue[raw_present & ~metrics_present] = "Raw data present but metrics missing"
    issue[~raw_present & metrics_present] = "Metrics present but raw data missing"

    report = df.loc[issue.notna(), ["Participant Id", "Date"]].copy()
    report["Stream"] = stream_name
    report["Issue"] = issue[issue.notna()].values

    return report

This chunk runs the validation function across all the streams and combines the result into a single report . It also summarizes the number of inconsistencies that were detected.

In [10]:
reports = []

for stream_name, config in STREAMS.items():
    stream_report = check_stream(
        df,
        stream_name,
        config["raw_col"],
        config["metric_cols"]
    )
    reports.append(stream_report)

final_report = pd.concat(reports, ignore_index=True)

print(final_report.shape)
print(final_report)

(501, 4)
    Participant Id        Date Stream                                 Issue
0         163lrwb6  2023-05-17    GPS  Raw data present but metrics missing
1         1akm4vyi  2023-05-14    GPS  Raw data present but metrics missing
2         9fn8c4k9  2024-03-23    GPS  Raw data present but metrics missing
3         bir3qzk4  2022-08-10    GPS  Raw data present but metrics missing
4         cqbxn7in  2023-06-01    GPS  Raw data present but metrics missing
..             ...         ...    ...                                   ...
496       zvbn42zg  2023-04-28  Texts  Metrics present but raw data missing
497       zvbn42zg  2023-04-29  Texts  Metrics present but raw data missing
498       zxie5nmv  2023-04-29  Texts  Metrics present but raw data missing
499       zxie5nmv  2023-05-06  Texts  Metrics present but raw data missing
500       zxie5nmv  2023-05-08  Texts  Metrics present but raw data missing

[501 rows x 4 columns]


Ensuring Expected Columns are Present

In [13]:
expected_columns = [
    "Date",
    "Participant Id",
    "Study Id",
    "Timezone",
    "Accelerometer Bytes",
    "Ambient Audio Bytes",
    "App Log Bytes",
    "Bluetooth Bytes",
    "Calls Bytes",
    "Devicemotion Bytes",
    "Gps Bytes",
    "Gyro Bytes",
    "Identifiers Bytes",
    "Ios Log Bytes",
    "Magnetometer Bytes",
    "Power State Bytes",
    "Proximity Bytes",
    "Reachability Bytes",
    "Survey Answers Bytes",
    "Survey Timings Bytes",
    "Texts Bytes",
    "Audio Recordings Bytes",
    "Wifi Bytes",
    "Distance Diameter",
    "Distance From Home",
    "Distance Traveled",
    "Flight Distance Average",
    "Flight Distance Stddev",
    "Flight Duration Average",
    "Flight Duration Stddev",
    "Home Duration",
    "Gyration Radius",
    "Significant Location Count",
    "Significant Location Entropy",
    "Pause Time",
    "Obs Duration",
    "Obs Day",
    "Obs Night",
    "Total Flight Time",
    "Av Pause Duration",
    "Sd Pause Duration",
    "Physical Circadian Rhythm",
    "Physical Circadian Rhythm Stratified",
    "Incoming Text Count",
    "Incoming Text Degree",
    "Incoming Text Length",
    "Outgoing Text Count",
    "Outgoing Text Degree",
    "Outgoing Text Length",
    "Incoming Text Reciprocity",
    "Outgoing Text Reciprocity",
    "Outgoing Mms Count",
    "Incoming Mms Count",
    "Mean Responsiveness Text",
    "Incoming Call Count",
    "Incoming Call Degree",
    "Incoming Call Duration",
    "Outgoing Call Count",
    "Outgoing Call Degree",
    "Outgoing Call Duration",
    "Missed Call Count",
    "Missed Callers",
    "Mean Responsiveness Call",
    "Call Reciprocity",
    "Uniq Individual Call Or Text Count",
    "Walking Time",
    "Steps",
    "Cadence"
]

missing_columns = [col for col in expected_columns if col not in df.columns]
extra_columns = [col for col in df.columns if col not in expected_columns]

print("===== Column Schema Check =====")

if not missing_columns and not extra_columns:
    print("PASS: All expected columns are present.")
else:
    if missing_columns:
        print("\nMissing columns:")
        for col in missing_columns:
            print(f"  - {col}")

    if extra_columns:
        print("\nUnexpected columns:")
        for col in extra_columns:
            print(f"  - {col}")

===== Column Schema Check =====
PASS: All expected columns are present.


Data Type Validation - Numeric

In [15]:
# Columns that should be numeric
numeric_columns = [
    col for col in df.columns
    if (
        "Bytes" in col
        or "Distance" in col
        or "Duration" in col
        or "Radius" in col
        or "Count" in col
        or "Entropy" in col
        or "Degree" in col
        or "Length" in col
        or "Reciprocity" in col
        or "Steps" in col
        or "Cadence" in col
        or "Responsiveness" in col
        or "Circadian" in col
    )
]

incorrect_types = []

for col in numeric_columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        incorrect_types.append((col, str(df[col].dtype)))

print("===== Data Type Validation =====")

if not incorrect_types:
    print("PASS: All numeric columns have numeric data types.")
else:
    print("FAIL: The following columns have incorrect data types:")
    for col, dtype in incorrect_types:
        print(f"  - {col}: {dtype}")

===== Data Type Validation =====
PASS: All numeric columns have numeric data types.


Range Checks - Non-Negative Values

In [16]:
negative_value_summary = {}

for col in numeric_columns:
    negative_count = (df[col] < 0).sum()
    
    if negative_count > 0:
        negative_value_summary[col] = negative_count

print("===== Range Checks: Non-negative Values =====")

if not negative_value_summary:
    print("PASS: No negative values found in numeric metric columns.")
else:
    print("FAIL: Negative values found:")
    for col, count in negative_value_summary.items():
        print(f"  - {col}: {count} negative values")

===== Range Checks: Non-negative Values =====
PASS: No negative values found in numeric metric columns.


The following chunk may be run if the range check failed and negative values are found. It will show which rows have negative values.

In [17]:
# Show rows with negative values

for col in numeric_columns:
    bad_rows = df[df[col] < 0]
    
    if not bad_rows.empty:
        print(f"\nNegative values in {col}:")
        print(bad_rows[["Participant Id", "Date", col]])